In [21]:
!pip install -q timm torchvision opencv-python pillow matplotlib pandas

import os
import io
import math
import glob
import time
import random
from typing import Dict, Any, Tuple, List, Optional

import numpy as np
from PIL import Image, ImageFilter, ImageEnhance, ImageDraw
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF

# Set seeds
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] Using: {DEVICE} (GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'})")

[Device] Using: cuda (GPU: Tesla T4)


In [24]:
# ---------------------------------------------------------------------------
# 1. Dataset Auto-Detection under /kaggle/input
# ---------------------------------------------------------------------------
def find_kaggle_dataset() -> Tuple[Optional[str], Optional[str]]:
    input_base = "/kaggle/input"
    if not os.path.exists(input_base):
        input_base = "./"
        
    print(f"[Dataset] Scanning for images in: {input_base}...")
    valid_exts = ('*.jpg', '*.jpeg', '*.png', '*.webp')
    
    csv_files = glob.glob(os.path.join(input_base, "**", "*.csv"), recursive=True)
    csv_path = csv_files[0] if len(csv_files) > 0 else None
    
    for ext in valid_exts:
        found = glob.glob(os.path.join(input_base, "**", ext), recursive=True)
        if len(found) > 10:
            dataset_dir = os.path.dirname(found[0])
            print(f"[Dataset] Found {len(found)} images in: {dataset_dir}")
            return dataset_dir, csv_path
            
    print("[Dataset] No files found in /kaggle/input, generating synthetic demo dataset...")
    return None, None


# ---------------------------------------------------------------------------
# 2. Bidirectional Domain Adaptation Transforms
# ---------------------------------------------------------------------------
class WebcamDegradationTransform:
    """Simulates webcam blur, compression artifacts, and sensor noise"""
    def apply_motion_blur(self, img: Image.Image) -> Image.Image:
        return img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.8, 2.2)))

    def apply_compression(self, img: Image.Image) -> Image.Image:
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=random.randint(35, 70))
        buf.seek(0)
        return Image.open(buf).copy()

    def apply_noise_and_color(self, img: Image.Image) -> Image.Image:
        arr = np.array(img).astype(np.float32)
        noise = np.random.normal(0, random.uniform(6.0, 14.0), arr.shape)
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
        noisy_img = Image.fromarray(arr)
        
        enhancer = ImageEnhance.Color(noisy_img)
        noisy_img = enhancer.enhance(random.uniform(0.8, 1.25))
        enhancer_b = ImageEnhance.Brightness(noisy_img)
        return enhancer_b.enhance(random.uniform(0.85, 1.15))

    def __call__(self, img: Image.Image) -> Image.Image:
        if random.random() < 0.5:
            img = self.apply_motion_blur(img)
        if random.random() < 0.5:
            img = self.apply_compression(img)
        if random.random() < 0.4:
            img = self.apply_noise_and_color(img)
        return img


class BiometricAugmentationPipeline:
    def __init__(self, is_train: bool = True, target_size: int = 224):
        self.is_train = is_train
        self.target_size = target_size
        self.webcam_degrader = WebcamDegradationTransform()
        self.normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    def __call__(self, img: Image.Image) -> torch.Tensor:
        img = img.convert("RGB")
        if self.is_train:
            if random.random() < 0.5:
                img = self.webcam_degrader(img)
            if random.random() < 0.5:
                img = TF.hflip(img)
            if random.random() < 0.35:
                angle = random.uniform(-10, 10)
                img = img.rotate(angle, resample=Image.Resampling.BILINEAR)

        img = img.resize((self.target_size, self.target_size), Image.Resampling.BILINEAR)
        tensor_img = TF.to_tensor(img)
        return self.normalize(tensor_img)


# ---------------------------------------------------------------------------
# 3. Dataset Loader
# ---------------------------------------------------------------------------
class KaggleBiometricDataset(Dataset):
    def __init__(self, data_dir: Optional[str] = None, csv_file: Optional[str] = None, is_train: bool = True, split_ratio: float = 0.85):
        self.is_train = is_train
        self.samples: List[Tuple[Any, float]] = []
        self.transform = BiometricAugmentationPipeline(is_train=is_train, target_size=224)

        all_items: List[Tuple[str, float]] = []

        if csv_file and os.path.exists(csv_file):
            import pandas as pd
            df = pd.read_csv(csv_file)
            path_col = [c for c in df.columns if any(k in c.lower() for k in ['path', 'file', 'image'])][0]
            age_col = [c for c in df.columns if 'age' in c.lower()][0]
            for _, row in df.iterrows():
                fpath = str(row[path_col])
                if not os.path.isabs(fpath) and data_dir:
                    fpath = os.path.join(data_dir, fpath)
                try:
                    age = float(row[age_col])
                    if 0 <= age <= 100 and os.path.exists(fpath):
                        all_items.append((fpath, age))
                except Exception:
                    continue
        elif data_dir and os.path.exists(data_dir):
            valid_exts = ('*.jpg', '*.jpeg', '*.png', '*.webp')
            files = []
            for ext in valid_exts:
                files.extend(glob.glob(os.path.join(data_dir, ext)))
                files.extend(glob.glob(os.path.join(data_dir, '**', ext), recursive=True))

            for fpath in files:
                fname = os.path.basename(fpath)
                parts = fname.replace('-', '_').split('_')
                try:
                    age = float(parts[0])
                    if 0 <= age <= 100:
                        all_items.append((fpath, age))
                except (ValueError, IndexError):
                    continue

        if len(all_items) == 0:
            print(f"[Dataset] Generating synthetic facial demo items for testing...")
            self.samples = self._generate_synthetic(160 if is_train else 40)
        else:
            random.Random(42).shuffle(all_items)
            split_idx = int(len(all_items) * split_ratio)
            self.samples = all_items[:split_idx] if is_train else all_items[split_idx:]
            print(f"[Dataset] {'Train' if is_train else 'Validation'} split loaded: {len(self.samples)} images")

    def _generate_synthetic(self, count: int) -> List[Tuple[Any, float]]:
        synth_list = []
        for _ in range(count):
            age = float(np.random.uniform(2.0, 85.0))
            img = Image.new('RGB', (224, 224), color=(int(220 - age*0.4), int(200 - age*0.5), int(185 - age*0.6)))
            draw = ImageDraw.Draw(img)
            draw.ellipse([45, 35, 179, 190], fill=(int(235 - age*0.3), int(205 - age*0.4), int(190 - age*0.5)), outline=(60, 40, 30), width=2)
            draw.ellipse([70, 90, 95, 105], fill=(50, 40, 30))
            draw.ellipse([129, 90, 154, 105], fill=(50, 40, 30))
            draw.line([(112, 105), (108, 130), (116, 130)], fill=(120, 80, 70), width=2)
            draw.arc([85, 140, 139, 165], start=0, end=180, fill=(150, 50, 50), width=2)
            if age > 35:
                for w in range(int((age - 35) / 6)):
                    draw.line([(70, 55 + w * 6), (154, 55 + w * 6)], fill=(130, 90, 80), width=1)
            synth_list.append((img, age))
        return synth_list

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        item, age = self.samples[idx]
        pil_img = Image.open(item).convert('RGB') if isinstance(item, str) else item.copy()
        tensor_img = self.transform(pil_img)
        target_age = torch.tensor(age, dtype=torch.float32)
        return tensor_img, target_age

In [11]:
# ---------------------------------------------------------------------------
# 1. ConvNeXt-Tiny (Local Texture Stream)
# ---------------------------------------------------------------------------
class ConvNeXtBlock(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dwconv = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        self.pwconv1 = nn.Linear(dim, 4 * dim)
        self.act = nn.GELU()
        self.pwconv2 = nn.Linear(4 * dim, dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        shortcut = x
        x = self.dwconv(x)
        x = x.permute(0, 2, 3, 1)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        x = x.permute(0, 3, 1, 2)
        return shortcut + x


class ConvNeXtTinyStream(nn.Module):
    def __init__(self, in_chans: int = 3, dims: list = [96, 192, 384, 768], depths: list = [3, 3, 9, 3]):
        super().__init__()
        self.downsample_layers = nn.ModuleList()
        stem = nn.Sequential(nn.Conv2d(in_chans, dims[0], kernel_size=4, stride=4), nn.GroupNorm(1, dims[0]))
        self.downsample_layers.append(stem)
        for i in range(3):
            self.downsample_layers.append(nn.Sequential(
                nn.GroupNorm(1, dims[i]),
                nn.Conv2d(dims[i], dims[i+1], kernel_size=2, stride=2)
            ))
        self.stages = nn.ModuleList([
            nn.Sequential(*[ConvNeXtBlock(dim=dims[i]) for _ in range(depths[i])])
            for i in range(4)
        ])
        self.norm = nn.LayerNorm(dims[-1], eps=1e-6)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for i in range(4):
            x = self.downsample_layers[i](x)
            x = self.stages[i](x)
        x = x.mean([-2, -1])
        return self.norm(x)


# ---------------------------------------------------------------------------
# 2. Swin Transformer V2-Tiny (Global Structural Geometry Stream)
# ---------------------------------------------------------------------------
class SwinV2TinyStream(nn.Module):
    def __init__(self, in_chans: int = 3, embed_dim: int = 96):
        super().__init__()
        self.patch_embed = nn.Sequential(
            nn.Conv2d(in_chans, embed_dim, kernel_size=4, stride=4),
            nn.GroupNorm(1, embed_dim)
        )
        self.encoder = nn.Sequential(
            nn.Conv2d(embed_dim, embed_dim * 2, kernel_size=2, stride=2),
            nn.GELU(),
            nn.Conv2d(embed_dim * 2, embed_dim * 4, kernel_size=2, stride=2),
            nn.GELU(),
            nn.Conv2d(embed_dim * 4, 768, kernel_size=2, stride=2),
            nn.GELU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.norm = nn.LayerNorm(768)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.patch_embed(x)
        x = self.encoder(x)
        return self.norm(x.flatten(1))


# ---------------------------------------------------------------------------
# 3. Cross-Stream Feature Fusion & LDL Head
# ---------------------------------------------------------------------------
class FeatureFusionHead(nn.Module):
    def __init__(self, in_dim: int = 1536, fused_dim: int = 768, num_classes: int = 101):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(in_dim, in_dim), nn.Sigmoid())
        self.proj = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, 1024),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(1024, fused_dim),
            nn.LayerNorm(fused_dim)
        )
        self.head = nn.Sequential(
            nn.Linear(fused_dim, 512),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(512, num_classes)
        )

    def forward(self, c_feat: torch.Tensor, s_feat: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        concat = torch.cat([c_feat, s_feat], dim=-1)
        gated = concat * self.gate(concat)
        fused = self.proj(gated)
        logits = self.head(fused)
        return logits, self.gate(concat)


class HybridAgePredictor(nn.Module):
    def __init__(self, num_classes: int = 101):
        super().__init__()
        self.num_classes = num_classes
        self.convnext = ConvNeXtTinyStream()
        self.swinv2 = SwinV2TinyStream()
        self.fusion = FeatureFusionHead(in_dim=1536, fused_dim=768, num_classes=num_classes)
        self.register_buffer("age_bins", torch.arange(0, num_classes, dtype=torch.float32))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        c_feat = self.convnext(x)
        s_feat = self.swinv2(x)
        logits, _ = self.fusion(c_feat, s_feat)
        return logits

    @torch.no_grad()
    def predict_age(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        self.eval()
        logits = self.forward(x)
        probs = F.softmax(logits, dim=-1)
        bins = self.age_bins.to(x.device)
        expected_age = (probs * bins.unsqueeze(0)).sum(dim=-1)
        diff_sq = (bins.unsqueeze(0) - expected_age.unsqueeze(1)) ** 2
        variance = (probs * diff_sq).sum(dim=-1)
        std_dev = torch.sqrt(variance)
        return expected_age, std_dev, probs


# ---------------------------------------------------------------------------
# 4. Label Distribution Learning (LDL) Loss
# ---------------------------------------------------------------------------
class LDLLoss(nn.Module):
    def __init__(self, num_classes: int = 101, sigma: float = 2.5, lambda_mae: float = 0.5):
        super().__init__()
        self.num_classes = num_classes
        self.sigma = sigma
        self.lambda_mae = lambda_mae
        self.register_buffer("bins", torch.arange(0, num_classes, dtype=torch.float32))

    def forward(self, logits: torch.Tensor, target_ages: torch.Tensor) -> Tuple[torch.Tensor, float, float]:
        bins = self.bins.to(logits.device).unsqueeze(0)
        target = target_ages.unsqueeze(1)
        
        gaussian_dist = torch.exp(-((bins - target) ** 2) / (2.0 * (self.sigma ** 2)))
        gaussian_dist = gaussian_dist / gaussian_dist.sum(dim=-1, keepdim=True)

        log_probs = F.log_softmax(logits, dim=-1)
        kl_loss = F.kl_div(log_probs, gaussian_dist, reduction='batchmean')

        pred_probs = torch.exp(log_probs)
        expected_age = (pred_probs * bins).sum(dim=-1)
        mae_loss = F.l1_loss(expected_age, target_ages)

        total_loss = kl_loss + self.lambda_mae * mae_loss
        return total_loss, kl_loss.item(), mae_loss.item()

In [20]:
def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = 15,
    lr: float = 1e-4,
    save_path: str = "/kaggle/working/best_hybrid_age_model.pt"
) -> Dict[str, List[float]]:
    criterion = LDLLoss(num_classes=101, sigma=2.5, lambda_mae=0.5).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

    history = {"train_loss": [], "val_mae": [], "val_cs3": [], "val_rmse": []}
    best_mae = float('inf')

    print(f"\n{'='*70}\nSTARTING HYBRID CONVNEXT + SWIN V2 LDL TRAINING ({epochs} EPOCHS)\n{'='*70}")

    for epoch in range(epochs):
        model.train()
        total_loss, total_kl, total_mae = 0.0, 0.0, 0.0
        start_t = time.time()

        for imgs, ages in train_loader:
            imgs, ages = imgs.to(DEVICE), ages.to(DEVICE)
            optimizer.zero_grad()

            if scaler is not None:
                with torch.amp.autocast('cuda'):
                    logits = model(imgs)
                    loss, kl_val, mae_val = criterion(logits, ages)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(imgs)
                loss, kl_val, mae_val = criterion(logits, ages)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item()
            total_kl += kl_val
            total_mae += mae_val

        scheduler.step()
        epoch_time = time.time() - start_t
        n_train = len(train_loader)

        # Validation
        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for imgs, ages in val_loader:
                imgs = imgs.to(DEVICE)
                preds, _, _ = model.predict_age(imgs)
                val_preds.extend(preds.cpu().tolist())
                val_targets.extend(ages.tolist())

        val_preds_t = torch.tensor(val_preds)
        val_targets_t = torch.tensor(val_targets)
        abs_err = torch.abs(val_preds_t - val_targets_t)
        v_mae = float(abs_err.mean().item())
        v_rmse = float(torch.sqrt((abs_err ** 2).mean()).item())
        v_cs3 = float((abs_err <= 3.0).sum().item() / len(abs_err) * 100.0)

        history["train_loss"].append(total_loss / n_train)
        history["val_mae"].append(v_mae)
        history["val_cs3"].append(v_cs3)
        history["val_rmse"].append(v_rmse)

        print(f"Epoch [{epoch+1:02d}/{epochs:02d}] "
              f"Train Loss: {total_loss/n_train:.4f} (KL: {total_kl/n_train:.3f}) | "
              f"Val MAE: {v_mae:.2f} yrs | CS@3: {v_cs3:.1f}% | RMSE: {v_rmse:.2f} | Time: {epoch_time:.1f}s")

        if v_mae < best_mae:
            best_mae = v_mae
            os.makedirs(os.path.dirname(save_path) if os.path.dirname(save_path) else ".", exist_ok=True)
            torch.save(model.state_dict(), save_path)
            print(f"  -> Checkpoint Saved: Val MAE {v_mae:.2f} yrs -> {save_path}")

    print(f"\n[Done] Training completed! Best Validation MAE: {best_mae:.2f} years.")
    return history

In [28]:
"""
================================================================================
FACIAL AGE PREDICTION — HYBRID CONVNEXT-TINY + SWIN TRANSFORMER V2-TINY MODEL
Using Gaussian Label Distribution Learning (LDL) & Bidirectional Domain Adaptation
Target: ~3.0 Years MAE across Webcam Frames & Clean Uploads (224x224 Native)
================================================================================
"""

!pip install -q timm torchvision opencv-python pillow pandas tqdm matplotlib

import os
import io
import math
import random
import numpy as np
import pandas as pd
from PIL import Image, ImageFilter, ImageEnhance
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as tv_models

try:
    import timm
    HAS_TIMM = True
except ImportError:
    HAS_TIMM = False

# -----------------------------------------------------------------------------
# 1. CONFIGURATION & HYPERPARAMETERS
# -----------------------------------------------------------------------------
DATA_DIR = "/kaggle/input/notebooks/vidhushinikg/age-prediction-preprocessing/processed_faces_10_80"

if not os.path.exists(DATA_DIR):
    for candidate in [
        "/kaggle/input/datasets/subisamayasundaram/preprocessed/processed_faces_10_80",
        "/kaggle/input/processed_faces_10_80",
        os.path.join(os.getcwd(), "processed_faces_10_80")
    ]:
        if os.path.exists(candidate):
            DATA_DIR = candidate
            break

MIN_AGE = 10
MAX_AGE = 80
NUM_CLASSES = MAX_AGE - MIN_AGE + 1  # 71 classes (ages 10 to 80 inclusive)
SIGMA = 2.5                          # Standard deviation for Gaussian label distribution

EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-2
LAMBDA_MAE = 0.5                     # Expectation MAE auxiliary loss weight
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("="*70)
print(f"🚀 [INFO] Device          : {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"📂 [INFO] Data Directory  : {DATA_DIR}")
print(f"🎯 [INFO] Age Range       : {MIN_AGE} - {MAX_AGE} ({NUM_CLASSES} LDL bins)")
print(f"⚙️  [INFO] Epochs / Batch  : {EPOCHS} epochs | batch size {BATCH_SIZE}")
print("="*70)

# -----------------------------------------------------------------------------
# 2. BIDIRECTIONAL DOMAIN-GAP AUGMENTATIONS
# -----------------------------------------------------------------------------
class WebcamSimulationAugmentation:
    def apply_motion_blur(self, img: Image.Image) -> Image.Image:
        return img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.8, 2.0)))

    def apply_compression(self, img: Image.Image) -> Image.Image:
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=random.randint(40, 75))
        buf.seek(0)
        return Image.open(buf).copy()

    def apply_lighting_shift(self, img: Image.Image) -> Image.Image:
        enhancer = ImageEnhance.Color(img)
        img = enhancer.enhance(random.uniform(0.8, 1.25))
        enhancer_b = ImageEnhance.Brightness(img)
        return enhancer_b.enhance(random.uniform(0.85, 1.15))

    def __call__(self, img: Image.Image) -> Image.Image:
        if random.random() < 0.45: img = self.apply_motion_blur(img)
        if random.random() < 0.45: img = self.apply_compression(img)
        if random.random() < 0.40: img = self.apply_lighting_shift(img)
        return img


def get_transforms():
    webcam_sim = WebcamSimulationAugmentation()

    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Lambda(lambda img: webcam_sim(img)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=8),
        transforms.ColorJitter(brightness=0.15, contrast=0.15),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    return train_transform, val_transform

# -----------------------------------------------------------------------------
# 3. GAUSSIAN LABEL DISTRIBUTION LEARNING (LDL) & EXPECTATION LOSS
# -----------------------------------------------------------------------------
def make_gaussian_label_distribution(ages: torch.Tensor, sigma: float = SIGMA, 
                                      min_age: int = MIN_AGE, num_classes: int = NUM_CLASSES, 
                                      device: str = DEVICE) -> torch.Tensor:
    class_indices = torch.arange(min_age, min_age + num_classes, device=device).float()  # (71,)
    diff = class_indices.unsqueeze(0) - ages.unsqueeze(1).float()                        # (B, 71)
    gaussian_dist = torch.exp(- (diff ** 2) / (2 * (sigma ** 2)))
    
    # Normalize rows to sum to 1.0
    dist_sum = torch.sum(gaussian_dist, dim=1, keepdim=True)
    return gaussian_dist / (dist_sum + 1e-8)


def ldl_combined_loss(logits: torch.Tensor, target_dist: torch.Tensor, target_ages: torch.Tensor,
                      min_age: int = MIN_AGE, num_classes: int = NUM_CLASSES, lambda_mae: float = LAMBDA_MAE,
                      device: str = DEVICE) -> Tuple[torch.Tensor, float, float]:
    log_probs = F.log_softmax(logits, dim=1)
    kl_loss = F.kl_div(log_probs, target_dist, reduction='batchmean')
    
    probs = torch.exp(log_probs)
    class_indices = torch.arange(min_age, min_age + num_classes, device=device).float()
    expected_age = torch.sum(probs * class_indices, dim=1)
    
    mae_loss = F.l1_loss(expected_age, target_ages)
    total_loss = kl_loss + lambda_mae * mae_loss
    return total_loss, kl_loss.item(), mae_loss.item()


def expected_age_class(logits: torch.Tensor, min_age: int = MIN_AGE, 
                       num_classes: int = NUM_CLASSES, device: str = DEVICE) -> Tuple[torch.Tensor, torch.Tensor]:
    probs = F.softmax(logits, dim=1)
    class_indices = torch.arange(min_age, min_age + num_classes, device=device).float()
    expected_age = torch.sum(probs * class_indices, dim=1)
    
    diff_sq = (class_indices.unsqueeze(0) - expected_age.unsqueeze(1)) ** 2
    variance = torch.sum(probs * diff_sq, dim=1)
    return expected_age, torch.sqrt(variance)

# -----------------------------------------------------------------------------
# 4. HYBRID CONVNEXT-TINY + SWIN V2-TINY MODEL (224x224 NATIVE)
# -----------------------------------------------------------------------------
class HybridConvNeXtSwinV2Model(nn.Module):
    """
    - ConvNeXt-Tiny: Extracts high-frequency local textures (wrinkles, skin elasticity).
    - Swin Transformer V2-Tiny (or Swin-T): Extracts global structural geometry.
    - Gated Feature Fusion Head: Concatenates both embeddings (1536-d) with cross-attention gating.
    """
    def __init__(self, num_classes=NUM_CLASSES, pretrained=True):
        super(HybridConvNeXtSwinV2Model, self).__init__()
        
        # 1. ConvNeXt-Tiny Local Stream
        print("[MODEL] Loading ConvNeXt-Tiny Local Feature Stream...")
        weights_conv = tv_models.ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None
        self.convnext_branch = tv_models.convnext_tiny(weights=weights_conv)
        conv_dim = self.convnext_branch.classifier[2].in_features  # 768
        self.convnext_branch.classifier = nn.Identity()

        # 2. Swin Transformer V2-Tiny Global Stream (224x224 Native)
        print("[MODEL] Loading Swin Transformer V2-Tiny Global Structural Stream...")
        swin_dim = 768
        
        # Priority: torchvision swin_v2_t / swin_t (native 224x224)
        if hasattr(tv_models, 'swin_v2_t'):
            try:
                weights_swinv2 = tv_models.Swin_V2_T_Weights.DEFAULT if pretrained else None
                self.swin_branch = tv_models.swin_v2_t(weights=weights_swinv2)
                swin_dim = self.swin_branch.head.in_features
                self.swin_branch.head = nn.Identity()
            except Exception:
                weights_swin = tv_models.Swin_T_Weights.DEFAULT if pretrained else None
                self.swin_branch = tv_models.swin_t(weights=weights_swin)
                swin_dim = self.swin_branch.head.in_features
                self.swin_branch.head = nn.Identity()
        elif HAS_TIMM:
            try:
                self.swin_branch = timm.create_model('swin_tiny_patch4_window7_224', pretrained=pretrained, num_classes=0)
                swin_dim = self.swin_branch.num_features
            except Exception:
                self.swin_branch = timm.create_model('swinv2_tiny_window16_256', pretrained=pretrained, num_classes=0, img_size=224)
                swin_dim = self.swin_branch.num_features
        else:
            weights_swin = tv_models.Swin_T_Weights.DEFAULT if pretrained else None
            self.swin_branch = tv_models.swin_t(weights=weights_swin)
            swin_dim = self.swin_branch.head.in_features
            self.swin_branch.head = nn.Identity()

        fusion_in_dim = conv_dim + swin_dim  # 1536
        
        # 3. Gated Cross-Stream Feature Fusion Block
        self.gate = nn.Sequential(
            nn.Linear(fusion_in_dim, fusion_in_dim),
            nn.Sigmoid()
        )
        
        self.fusion_mlp = nn.Sequential(
            nn.LayerNorm(fusion_in_dim),
            nn.Linear(fusion_in_dim, 768),
            nn.GELU(),
            nn.Dropout(p=0.25),
            nn.LayerNorm(768),
            nn.Linear(768, 512),
            nn.GELU(),
            nn.Dropout(p=0.15),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        feat_conv = self.convnext_branch(x)  # (B, 768)
        feat_swin = self.swin_branch(x)      # (B, 768)
        
        if feat_conv.dim() > 2:
            feat_conv = torch.flatten(feat_conv, 1)
        if feat_swin.dim() > 2:
            feat_swin = torch.flatten(feat_swin, 1)
            
        concat_feats = torch.cat([feat_conv, feat_swin], dim=1)  # (B, 1536)
        gated_feats = concat_feats * self.gate(concat_feats)
        logits = self.fusion_mlp(gated_feats)                    # (B, 71)
        return logits

# -----------------------------------------------------------------------------
# 5. DATASET & DATALOADERS
# -----------------------------------------------------------------------------
class FaceAgeDataset(Dataset):
    def __init__(self, df: pd.DataFrame, root_dir: str, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        
        self.path_col = next((c for c in ['filepath', 'filename', 'img_name', 'image_path', 'path', 'file'] if c in self.df.columns), self.df.columns[0])
        self.age_col = next((c for c in ['age', 'Age', 'target', 'label', 'real_age'] if c in self.df.columns), self.df.columns[1])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        val_path = str(row[self.path_col])
        
        if os.path.exists(val_path):
            img_path = val_path
        else:
            img_path = os.path.join(self.root_dir, os.path.basename(val_path))
            if not os.path.exists(img_path):
                img_path = os.path.join(self.root_dir, val_path)

        image = Image.open(img_path).convert("RGB")
        age = float(row[self.age_col])

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(age, dtype=torch.float32)

# -----------------------------------------------------------------------------
# 6. METRICS CALCULATION (MAE, RMSE, CS@1, CS@3, CS@5)
# -----------------------------------------------------------------------------
def compute_metrics(preds: np.ndarray, targets: np.ndarray):
    diff = np.abs(preds - targets)
    mae = np.mean(diff)
    rmse = np.sqrt(np.mean((preds - targets) ** 2))
    cs_1 = np.mean(diff <= 1.0) * 100.0
    cs_3 = np.mean(diff <= 3.0) * 100.0
    cs_5 = np.mean(diff <= 5.0) * 100.0
    return {"mae": mae, "rmse": rmse, "cs_1": cs_1, "cs_3": cs_3, "cs_5": cs_5}

# -----------------------------------------------------------------------------
# 7. TRAINING & EVALUATION PIPELINE
# -----------------------------------------------------------------------------
def train_and_evaluate():
    train_csv = os.path.join(DATA_DIR, "train.csv")
    val_csv = os.path.join(DATA_DIR, "val.csv")
    test_csv = os.path.join(DATA_DIR, "test.csv")

    if not (os.path.exists(train_csv) and os.path.exists(val_csv)):
        parent_dir = os.path.dirname(DATA_DIR)
        if os.path.exists(os.path.join(parent_dir, "train.csv")):
            train_csv = os.path.join(parent_dir, "train.csv")
            val_csv = os.path.join(parent_dir, "val.csv")
            test_csv = os.path.join(parent_dir, "test.csv")

    train_df = pd.read_csv(train_csv)
    val_df = pd.read_csv(val_csv)
    test_df = pd.read_csv(test_csv) if os.path.exists(test_csv) else None

    train_df = train_df[(train_df['age'] >= MIN_AGE) & (train_df['age'] <= MAX_AGE)]
    val_df = val_df[(val_df['age'] >= MIN_AGE) & (val_df['age'] <= MAX_AGE)]
    if test_df is not None:
        test_df = test_df[(test_df['age'] >= MIN_AGE) & (test_df['age'] <= MAX_AGE)]

    print(f"\n📊 [DATA] Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df) if test_df is not None else 0:,}")

    train_tf, val_tf = get_transforms()
    train_ds = FaceAgeDataset(train_df, DATA_DIR, transform=train_tf)
    val_ds = FaceAgeDataset(val_df, DATA_DIR, transform=val_tf)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

    # Initialize Hybrid Model
    model = HybridConvNeXtSwinV2Model(num_classes=NUM_CLASSES, pretrained=True).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == "cuda"))

    best_val_mae = float('inf')
    best_model_path = "/kaggle/working/best_hybrid_convnext_swinv2_model.pth"

    print("\n" + "="*70)
    print("🚀 STARTING TRAINING (10 EPOCHS WITH CONVNEXT + SWIN V2 LDL)")
    print("="*70)

    for epoch in range(EPOCHS):
        model.train()
        train_loss, train_mae_sum = 0.0, 0.0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/{EPOCHS:02d} [Train]")
        for imgs, ages in pbar:
            imgs, ages = imgs.to(DEVICE), ages.to(DEVICE)
            target_dist = make_gaussian_label_distribution(ages, device=DEVICE)

            optimizer.zero_grad()
            with torch.amp.autocast('cuda', enabled=(DEVICE == "cuda")):
                logits = model(imgs)
                loss, kl_val, mae_val = ldl_combined_loss(logits, target_dist, ages, device=DEVICE)

            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * imgs.size(0)
            preds, _ = expected_age_class(logits, device=DEVICE)
            train_mae_sum += torch.abs(preds - ages).sum().item()

            pbar.set_postfix({"loss": f"{loss.item():.4f}", "kl": f"{kl_val:.2f}", "mae": f"{mae_val:.2f}"})

        scheduler.step()

        train_loss /= len(train_ds)
        train_mae = train_mae_sum / len(train_ds)

        # Validation Phase
        model.eval()
        val_loss = 0.0
        val_preds_list, val_ages_list = [], []

        with torch.no_grad():
            for imgs, ages in val_loader:
                imgs, ages = imgs.to(DEVICE), ages.to(DEVICE)
                target_dist = make_gaussian_label_distribution(ages, device=DEVICE)

                with torch.amp.autocast('cuda', enabled=(DEVICE == "cuda")):
                    logits = model(imgs)
                    loss, _, _ = ldl_combined_loss(logits, target_dist, ages, device=DEVICE)

                val_loss += loss.item() * imgs.size(0)
                preds, _ = expected_age_class(logits, device=DEVICE)

                val_preds_list.extend(preds.cpu().numpy())
                val_ages_list.extend(ages.cpu().numpy())

        val_loss /= len(val_ds)
        val_metrics = compute_metrics(np.array(val_preds_list), np.array(val_ages_list))
        val_mae = val_metrics["mae"]

        print(f"👉 [Epoch {epoch+1:02d}] Train Loss: {train_loss:.4f} (MAE: {train_mae:.2f}y) | "
              f"Val MAE: {val_mae:.2f} yrs | CS@3: {val_metrics['cs_3']:.1f}% | CS@5: {val_metrics['cs_5']:.1f}%")

        # Save Best Model Checkpoint
        if val_mae < best_val_mae:
            best_val_mae = val_mae
            os.makedirs(os.path.dirname(best_model_path), exist_ok=True)
            torch.save(model.state_dict(), best_model_path)
            print(f"   💾 --> Best Model Saved! Improved Val MAE to {best_val_mae:.2f} years")

    print("\n" + "="*70)
    print(f"🎉 TRAINING COMPLETE! Best Validation MAE: {best_val_mae:.2f} years")
    print(f"💾 Checkpoint stored at: {best_model_path}")
    print("="*70)

    # Final Test Set Evaluation
    if test_df is not None:
        print("\n[EVAL] Evaluating Best Checkpoint on Test Set...")
        model.load_state_dict(torch.load(best_model_path))
        model.eval()

        test_ds = FaceAgeDataset(test_df, DATA_DIR, transform=val_tf)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

        test_preds, test_ages = [], []
        with torch.no_grad():
            for imgs, ages in tqdm(test_loader, desc="Testing"):
                imgs = imgs.to(DEVICE)
                logits = model(imgs)
                preds, _ = expected_age_class(logits, device=DEVICE)
                test_preds.extend(preds.cpu().numpy())
                test_ages.extend(ages.numpy())

        test_metrics = compute_metrics(np.array(test_preds), np.array(test_ages))
        print("\n" + "="*60)
        print("🏆 FINAL BENCHMARK TEST SET EVALUATION:")
        print(f" • Test MAE (Mean Absolute Error)     : {test_metrics['mae']:.2f} years")
        print(f" • Test RMSE                           : {test_metrics['rmse']:.2f} years")
        print(f" • Cumulative Score CS@1 (<= 1 year)  : {test_metrics['cs_1']:.2f}%")
        print(f" • Cumulative Score CS@3 (<= 3 years) : {test_metrics['cs_3']:.2f}% (Target: ~80%)")
        print(f" • Cumulative Score CS@5 (<= 5 years) : {test_metrics['cs_5']:.2f}%")
        print("="*60)

if __name__ == "__main__":
    train_and_evaluate()

🚀 [INFO] Device          : cuda (Tesla T4)
📂 [INFO] Data Directory  : /kaggle/input/datasets/subisamayasundaram/preprocessed/processed_faces_10_80
🎯 [INFO] Age Range       : 10 - 80 (71 LDL bins)
⚙️  [INFO] Epochs / Batch  : 10 epochs | batch size 32

📊 [DATA] Train: 28,107 | Val: 6,023 | Test: 6,023
[MODEL] Loading ConvNeXt-Tiny Local Feature Stream...
[MODEL] Loading Swin Transformer V2-Tiny Global Structural Stream...
Downloading: "https://download.pytorch.org/models/swin_v2_t-b137f0e2.pth" to /root/.cache/torch/hub/checkpoints/swin_v2_t-b137f0e2.pth


100%|██████████| 109M/109M [00:02<00:00, 40.1MB/s] 



🚀 STARTING TRAINING (10 EPOCHS WITH CONVNEXT + SWIN V2 LDL)


Epoch 01/10 [Train]:   0%|          | 0/879 [00:00<?, ?it/s]

👉 [Epoch 01] Train Loss: 8.4339 (MAE: 13.21y) | Val MAE: 16.04 yrs | CS@3: 11.1% | CS@5: 18.8%
   💾 --> Best Model Saved! Improved Val MAE to 16.04 years


...

👉 [Epoch 09] Train Loss: 8.1685 (MAE: 12.69y) | Val MAE: 12.51 yrs | CS@3: 15.1% | CS@5: 25.0%


Epoch 10/10 [Train]:   0%|          | 0/879 [00:00<?, ?it/s]

👉 [Epoch 10] Train Loss: 7.9968 (MAE: 12.43y) | Val MAE: 12.35 yrs | CS@3: 15.5% | CS@5: 25.8%
   💾 --> Best Model Saved! Improved Val MAE to 12.35 years

🎉 TRAINING COMPLETE! Best Validation MAE: 12.35 years
💾 Checkpoint stored at: /kaggle/working/best_hybrid_convnext_swinv2_model.pth

[EVAL] Evaluating Best Checkpoint on Test Set...


Testing:   0%|          | 0/189 [00:00<?, ?it/s]


🏆 FINAL BENCHMARK TEST SET EVALUATION:
 • Test MAE (Mean Absolute Error)     : 12.02 years
 • Test RMSE                           : 15.37 years
 • Cumulative Score CS@1 (<= 1 year)  : 5.36%
 • Cumulative Score CS@3 (<= 3 years) : 16.07% (Target: ~80%)
 • Cumulative Score CS@5 (<= 5 years) : 26.56%


In [6]:
import os, re

DATA_DIR = "/kaggle/input/datasets/subisamayasundaram/preprocessed/processed_faces_10_80"
MIN_AGE, MAX_AGE = 10, 80
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def parse_age_from_path(filepath):
    fname = os.path.splitext(os.path.basename(filepath))[0]
    tokens = re.split(r'[_\-\s]', fname)
    for token in reversed(tokens):
        if token.isdigit():
            age = int(token)
            if MIN_AGE <= age <= MAX_AGE:
                return age
    parent = os.path.basename(os.path.dirname(filepath))
    if parent.isdigit():
        age = int(parent)
        if MIN_AGE <= age <= MAX_AGE:
            return age
    return None

count, skipped = 0, 0
for dirpath, _, filenames in os.walk(DATA_DIR):
    for fname in filenames:
        if os.path.splitext(fname)[1].lower() not in SUPPORTED_EXTENSIONS:
            continue
        age = parse_age_from_path(os.path.join(dirpath, fname))
        if age is not None:
            count += 1
        else:
            skipped += 1

print(f"✅ Valid images parsed : {count:,}")
print(f"⚠️  Skipped (no age)   : {skipped:,}")

Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 259, in _feed
    reader_close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 178, in close
    self._close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 377, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 259, in _feed
    reader_close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 178, in close
    self._close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 377, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 259, in _feed
    reader_close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 178, in close
    self._close()
  File "/usr/lib/python3.12/multip

✅ Valid images parsed : 40,153
⚠️  Skipped (no age)   : 0


In [9]:
# Cell 1: Clear stale cached splits
import os
for f in ["/kaggle/working/train.csv", "/kaggle/working/val.csv", "/kaggle/working/test.csv"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted: {f}")
print("Done. Now rerun the training cell.")

Done. Now rerun the training cell.


In [ ]:
"""
================================================================================
FACIAL AGE PREDICTION — SOTA CONVNEXT-SMALL + SWIN-T HYBRID ENGINE
Architecture: ConvNeXt-Small + Swin-T with Squeeze-and-Excitation Cross-Gating
Loss Engine : Compound Triple Loss (Gaussian LDL + Smooth L1 + Variance Loss)
Optimization: Differential Learning Rates + OneCycleLR with Cosine Annealing
Inference   : Test-Time Augmentation (TTA with Horizontal Flip Ensembling)
Target      : Sub 4.0 Validation / Test MAE
================================================================================
"""

import os
import re
import math
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as tv_models

# =============================================================================
# 1. CONFIGURATION
# =============================================================================

DATA_DIR = "/kaggle/input/datasets/subisamayasundaram/preprocessed/processed_faces_10_80"

MIN_AGE    = 10
MAX_AGE    = 80
NUM_CLASSES = MAX_AGE - MIN_AGE + 1   # 71 age bins

SIGMA      = 2.0
LAMBDA_KL  = 1.0
LAMBDA_L1  = 1.5
LAMBDA_VAR = 0.05

EPOCHS     = 30
BATCH_SIZE = 32
LR_BACKBONE = 1e-4
LR_HEAD     = 5e-4
WEIGHT_DECAY = 1e-2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BEST_MODEL_PATH = "/kaggle/working/best_age_model_sota.pth"
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("=" * 70)
print("FACIAL AGE ESTIMATION — SOTA HYBRID ENGINE")
print("=" * 70)
print(f"[INFO] Device      : {DEVICE} ({torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'})")
print(f"[INFO] Data Dir    : {DATA_DIR}")
print(f"[INFO] Age Range   : {MIN_AGE}-{MAX_AGE} ({NUM_CLASSES} bins)")
print(f"[INFO] Epochs      : {EPOCHS} | Batch: {BATCH_SIZE}")

# =============================================================================
# 2. AUTO DATASET SCANNER & SPLIT GENERATOR
# =============================================================================

def parse_age_from_path(filepath: str) -> int | None:
    """
    Parses age from {id}_{age}.jpg format (e.g. 2936_15.jpg → 15).
    Falls back to parent folder name if it is a digit.
    """
    fname = os.path.splitext(os.path.basename(filepath))[0]

    # Primary: last numeric token e.g. 2936_15 → ['2936','15'] → age=15
    for token in reversed(re.split(r'[_\-\s]', fname)):
        if token.isdigit():
            age = int(token)
            if MIN_AGE <= age <= MAX_AGE:
                return age

    # Fallback: parent folder is a digit e.g. .../25/img.jpg
    parent = os.path.basename(os.path.dirname(filepath))
    if parent.isdigit():
        age = int(parent)
        if MIN_AGE <= age <= MAX_AGE:
            return age

    return None


def build_dataframe_from_directory(root_dir: str) -> pd.DataFrame:
    print(f"[SCAN] Scanning: {root_dir}")
    records = []
    for dirpath, _, filenames in os.walk(root_dir):
        for fname in filenames:
            if os.path.splitext(fname)[1].lower() not in SUPPORTED_EXTENSIONS:
                continue
            full_path = os.path.join(dirpath, fname)
            age = parse_age_from_path(full_path)
            if age is None:
                continue
            rel_path = os.path.relpath(full_path, root_dir)
            records.append({"filename": rel_path, "age": age})
    df = pd.DataFrame(records)
    print(f"[SCAN] Found {len(df):,} valid images | Age range: {df['age'].min()}-{df['age'].max()}")
    return df


# Load or build splits
train_csv = "/kaggle/working/train.csv"
val_csv   = "/kaggle/working/val.csv"
test_csv  = "/kaggle/working/test.csv"

if os.path.exists(train_csv) and os.path.exists(val_csv):
    print("[DATA] Loading cached CSV splits from /kaggle/working/...")
    train_df = pd.read_csv(train_csv)
    val_df   = pd.read_csv(val_csv)
    test_df  = pd.read_csv(test_csv) if os.path.exists(test_csv) else None
else:
    full_df = build_dataframe_from_directory(DATA_DIR)

    if len(full_df) == 0:
        raise RuntimeError(f"[ERROR] No valid images found in {DATA_DIR}.")

    full_df["age_bin"] = pd.cut(full_df["age"], bins=10, labels=False)

    train_df, temp_df = train_test_split(
        full_df, test_size=0.30, random_state=42, stratify=full_df["age_bin"]
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, random_state=42, stratify=temp_df["age_bin"]
    )

    for df_ in [train_df, val_df, test_df]:
        df_.drop(columns=["age_bin"], inplace=True, errors="ignore")

    train_df = train_df.reset_index(drop=True)
    val_df   = val_df.reset_index(drop=True)
    test_df  = test_df.reset_index(drop=True)

    os.makedirs("/kaggle/working", exist_ok=True)
    train_df.to_csv(train_csv, index=False)
    val_df.to_csv(val_csv,     index=False)
    test_df.to_csv(test_csv,   index=False)
    print("[DATA] Splits saved to /kaggle/working/")

print(f"[DATA] Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

# =============================================================================
# 3. COMPOUND LOSS
# =============================================================================

class CompoundAgeLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.smooth_l1 = nn.SmoothL1Loss(beta=1.0)

    def forward(self, logits, target_ages):
        class_indices = torch.arange(
            MIN_AGE, MIN_AGE + NUM_CLASSES, device=logits.device, dtype=torch.float32
        )

        # Gaussian target distribution
        diff = class_indices.unsqueeze(0) - target_ages.unsqueeze(1)
        target_dist = torch.exp(-(diff ** 2) / (2.0 * SIGMA ** 2))
        target_dist = target_dist / (target_dist.sum(dim=1, keepdim=True) + 1e-8)

        # KL Divergence
        log_probs = F.log_softmax(logits, dim=1)
        loss_kl = F.kl_div(log_probs, target_dist, reduction="batchmean")

        # Expected age + Smooth L1
        probs = torch.exp(log_probs)
        expected_age = torch.sum(probs * class_indices.unsqueeze(0), dim=1)
        loss_l1 = self.smooth_l1(expected_age, target_ages)

        # Variance penalty
        var_diff_sq = (class_indices.unsqueeze(0) - expected_age.unsqueeze(1)) ** 2
        loss_var = torch.mean(torch.sum(probs * var_diff_sq, dim=1))

        total = LAMBDA_KL * loss_kl + LAMBDA_L1 * loss_l1 + LAMBDA_VAR * loss_var
        return total, loss_kl.item(), loss_l1.item(), loss_var.item(), expected_age


def compute_expected_age(logits):
    probs = F.softmax(logits, dim=1)
    class_indices = torch.arange(
        MIN_AGE, MIN_AGE + NUM_CLASSES, device=logits.device, dtype=torch.float32
    )
    return torch.sum(probs * class_indices.unsqueeze(0), dim=1)

# =============================================================================
# 4. SQUEEZE-AND-EXCITATION FUSION
# =============================================================================

class SEFusion(nn.Module):
    def __init__(self, in_dim, reduction=16):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(in_dim, in_dim // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_dim // reduction, in_dim, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.gate(x)

# =============================================================================
# 5. HYBRID CONVNEXT-SMALL + SWIN-T MODEL
# =============================================================================

class HybridAgeModel(nn.Module):
    def __init__(self):
        super().__init__()

        # ConvNeXt-Small (local texture / wrinkle stream)
        print("[MODEL] Loading ConvNeXt-Small...")
        self.cnn_branch = tv_models.convnext_small(weights=tv_models.ConvNeXt_Small_Weights.DEFAULT)
        cnn_dim = self.cnn_branch.classifier[2].in_features   # 768
        self.cnn_branch.classifier[2] = nn.Identity()
        print(f"[MODEL] ConvNeXt-Small feature dim: {cnn_dim}")

        # Swin-T (global geometry / craniofacial structure stream)
        print("[MODEL] Loading Swin-T...")
        self.swin_branch = tv_models.swin_t(weights=tv_models.Swin_T_Weights.DEFAULT)
        swin_dim = self.swin_branch.head.in_features           # 768
        self.swin_branch.head = nn.Identity()
        print(f"[MODEL] Swin-T feature dim        : {swin_dim}")

        fusion_dim = cnn_dim + swin_dim  # 1536

        # SE-Gated fusion
        self.se_fusion = SEFusion(fusion_dim, reduction=16)

        # Classification head
        self.head = nn.Sequential(
            nn.BatchNorm1d(fusion_dim),
            nn.Dropout(p=0.35),
            nn.Linear(fusion_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(p=0.25),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(p=0.15),
            nn.Linear(256, NUM_CLASSES)
        )

    def forward(self, x):
        f_cnn  = self.cnn_branch(x)
        f_swin = self.swin_branch(x)
        if f_cnn.dim()  > 2: f_cnn  = torch.flatten(f_cnn,  1)
        if f_swin.dim() > 2: f_swin = torch.flatten(f_swin, 1)
        fused  = torch.cat([f_cnn, f_swin], dim=1)   # (B, 1536)
        return self.head(self.se_fusion(fused))

# =============================================================================
# 6. DATASET & TRANSFORMS
# =============================================================================

class FaceAgeDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.root_dir  = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        rel_path = str(row["filename"])
        img_path = rel_path if os.path.exists(rel_path) else os.path.join(self.root_dir, rel_path)

        image = Image.open(img_path).convert("RGB")
        age   = float(row["age"])

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(age, dtype=torch.float32)


def get_transforms():
    train_tf = transforms.Compose([
        transforms.Resize((236, 236)),
        transforms.RandomResizedCrop(224, scale=(0.88, 1.0), ratio=(0.95, 1.05)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.03),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.15, scale=(0.02, 0.15), value="random")
    ])
    val_tf = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return train_tf, val_tf

# =============================================================================
# 7. METRICS
# =============================================================================

def compute_metrics(preds, targets):
    diff = np.abs(preds - targets)
    return {
        "mae":  float(np.mean(diff)),
        "rmse": float(np.sqrt(np.mean((preds - targets) ** 2))),
        "cs_3": float(np.mean(diff <= 3.0) * 100.0),
        "cs_5": float(np.mean(diff <= 5.0) * 100.0),
    }

# =============================================================================
# 8. TRAINING PIPELINE
# =============================================================================

def train_and_evaluate():
    train_tf, val_tf = get_transforms()

    train_ds = FaceAgeDataset(train_df, DATA_DIR, transform=train_tf)
    val_ds   = FaceAgeDataset(val_df,   DATA_DIR, transform=val_tf)

    # num_workers=2 avoids multiprocessing crashes in Kaggle notebooks
    NUM_WORKERS = 2

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True,
                              persistent_workers=True, drop_last=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True,
                              persistent_workers=True)

    model     = HybridAgeModel().to(DEVICE)
    criterion = CompoundAgeLoss()

    # Differential learning rates: backbone vs head
    backbone_params = [p for n, p in model.named_parameters()
                       if ("cnn_branch" in n or "swin_branch" in n) and p.requires_grad]
    head_params     = [p for n, p in model.named_parameters()
                       if ("cnn_branch" not in n and "swin_branch" not in n) and p.requires_grad]

    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": LR_BACKBONE, "weight_decay": WEIGHT_DECAY},
        {"params": head_params,     "lr": LR_HEAD,     "weight_decay": 1e-3}
    ])

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=[LR_BACKBONE, LR_HEAD],
        total_steps=len(train_loader) * EPOCHS,
        pct_start=0.15,
        div_factor=15.0,
        final_div_factor=1000.0,
        anneal_strategy="cos"
    )

    scaler       = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))
    best_val_mae = float("inf")

    print("\n" + "=" * 70)
    print("STARTING TRAINING (Target: Sub 4.0 MAE)")
    print("=" * 70)

    for epoch in range(EPOCHS):

        # ── TRAIN ─────────────────────────────────────────────────────────────
        model.train()
        train_loss_sum, train_mae_sum = 0.0, 0.0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:02d}/{EPOCHS} [Train]")
        for imgs, ages in pbar:
            imgs = imgs.to(DEVICE, non_blocking=True)
            ages = ages.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
                logits                              = model(imgs)
                loss, kl, l1, var, pred_ages = criterion(logits, ages)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            bs = imgs.size(0)
            train_loss_sum += loss.item() * bs
            train_mae_sum  += torch.abs(pred_ages.detach() - ages).sum().item()

            pbar.set_postfix(loss=f"{loss.item():.3f}", l1=f"{l1:.2f}",
                             lr=f"{optimizer.param_groups[1]['lr']:.2e}")

        avg_train_loss = train_loss_sum / len(train_ds)
        avg_train_mae  = train_mae_sum  / len(train_ds)

        # ── VALIDATE (+ TTA) ──────────────────────────────────────────────────
        model.eval()
        val_loss_sum = 0.0
        val_preds, val_ages_list = [], []

        with torch.no_grad():
            for imgs, ages in tqdm(val_loader, desc=f"Epoch {epoch+1:02d}/{EPOCHS} [Val+TTA]"):
                imgs = imgs.to(DEVICE, non_blocking=True)
                ages = ages.to(DEVICE, non_blocking=True)

                with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
                    logits_orig = model(imgs)
                    logits_flip = model(torch.flip(imgs, dims=[3]))
                    logits      = 0.5 * (logits_orig + logits_flip)
                    loss, _, _, _, pred_ages = criterion(logits, ages)

                val_loss_sum += loss.item() * imgs.size(0)
                val_preds.extend(pred_ages.cpu().numpy())
                val_ages_list.extend(ages.cpu().numpy())

        avg_val_loss = val_loss_sum / len(val_ds)
        m = compute_metrics(np.array(val_preds), np.array(val_ages_list))

        print("-" * 70)
        print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
              f"Train Loss: {avg_train_loss:.4f} | Train MAE: {avg_train_mae:.2f} yrs | "
              f"Val Loss: {avg_val_loss:.4f} | Val MAE: {m['mae']:.2f} yrs | "
              f"CS@3: {m['cs_3']:.1f}% | CS@5: {m['cs_5']:.1f}%")

        if m["mae"] < best_val_mae:
            best_val_mae = m["mae"]
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            print(f" --> Best Model Saved! Val MAE improved to {best_val_mae:.2f} yrs")

    print("\n" + "=" * 70)
    print(f"TRAINING COMPLETE — Best Val MAE: {best_val_mae:.2f} years")
    print("=" * 70)

    # ── TEST EVALUATION ───────────────────────────────────────────────────────
    if test_df is not None:
        print("\n[EVAL] Running final test set evaluation with TTA...")
        model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE, weights_only=True))
        model.eval()

        test_ds     = FaceAgeDataset(test_df, DATA_DIR, transform=val_tf)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                                 num_workers=NUM_WORKERS, pin_memory=True,
                                 persistent_workers=True)

        test_loss_sum = 0.0
        test_preds, test_ages_list = [], []

        with torch.no_grad():
            for imgs, ages in tqdm(test_loader, desc="Testing"):
                imgs = imgs.to(DEVICE, non_blocking=True)
                ages_dev = ages.to(DEVICE, non_blocking=True)

                with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
                    logits_orig = model(imgs)
                    logits_flip = model(torch.flip(imgs, dims=[3]))
                    logits      = 0.5 * (logits_orig + logits_flip)
                    loss, _, _, _, pred_ages = criterion(logits, ages_dev)

                test_loss_sum += loss.item() * imgs.size(0)
                test_preds.extend(pred_ages.cpu().numpy())
                test_ages_list.extend(ages.numpy())

        avg_test_loss = test_loss_sum / len(test_ds)
        tm = compute_metrics(np.array(test_preds), np.array(test_ages_list))

        print("\n" + "=" * 60)
        print("FINAL TEST EVALUATION METRICS:")
        print(f" - Test Loss (Compound)            : {avg_test_loss:.4f}")
        print(f" - Test MAE (Mean Absolute Error)  : {tm['mae']:.2f} years")
        print(f" - Test RMSE                        : {tm['rmse']:.2f} years")
        print(f" - Cumulative Score CS@3 (<= 3 yrs): {tm['cs_3']:.2f}%")
        print(f" - Cumulative Score CS@5 (<= 5 yrs): {tm['cs_5']:.2f}%")
        print("=" * 60)


if __name__ == "__main__":
    train_and_evaluate()

FACIAL AGE ESTIMATION — SOTA HYBRID ENGINE
[INFO] Device      : cuda (Tesla T4)
[INFO] Data Dir    : /kaggle/input/datasets/subisamayasundaram/preprocessed/processed_faces_10_80
[INFO] Age Range   : 10-80 (71 bins)
[INFO] Epochs      : 30 | Batch: 32
[SCAN] Scanning: /kaggle/input/datasets/subisamayasundaram/preprocessed/processed_faces_10_80
[SCAN] Found 40,153 valid images | Age range: 10-80
[DATA] Splits saved to /kaggle/working/
[DATA] Train: 28,107 | Val: 6,023 | Test: 6,023
[MODEL] Loading ConvNeXt-Small...
[MODEL] ConvNeXt-Small feature dim: 768
[MODEL] Loading Swin-T...
[MODEL] Swin-T feature dim        : 768

STARTING TRAINING (Target: Sub 4.0 MAE)


Epoch 01/30 [Train]:   0%|          | 0/878 [00:00<?, ?it/s]

/tmp/ipykernel_59/946805088.py:388: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
Exception in thread QueueFeederThread:
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 259, in _feed
    reader_close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 178, in close
    self._close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 377, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py"

Epoch 01/30 [Val+TTA]:   0%|          | 0/189 [00:00<?, ?it/s]

----------------------------------------------------------------------
Epoch 01/30 | Train Loss: 28.4153 | Train MAE: 13.63 yrs | Val Loss: 23.4950 | Val MAE: 12.77 yrs | CS@3: 12.9% | CS@5: 21.7%
 --> Best Model Saved! Val MAE improved to 12.77 yrs


...

----------------------------------------------------------------------
Epoch 27/30 | Train Loss: 5.2216 | Train MAE: 3.05 yrs | Val Loss: 12.0763 | Val MAE: 6.62 yrs | CS@3: 37.8% | CS@5: 55.1%


Epoch 28/30 [Train]:   0%|          | 0/878 [00:00<?, ?it/s]

Epoch 28/30 [Val+TTA]:   0%|          | 0/189 [00:00<?, ?it/s]

----------------------------------------------------------------------
Epoch 28/30 | Train Loss: 5.1450 | Train MAE: 3.01 yrs | Val Loss: 12.0498 | Val MAE: 6.61 yrs | CS@3: 37.7% | CS@5: 55.8%


Epoch 29/30 [Train]:   0%|          | 0/878 [00:00<?, ?it/s]